In [57]:
from scone_tools.evaluation.reconstruction_evaluation import best_permutation_similarity


In [19]:
def simulate_views(n=600, M_G=20, M_C=20, 
                   rank=3, seed=0, gamma=1, noise=0,
                   sparsity=0, rG=0, rGC=0, rZ=0,
                   signed_cov_effects=False,subgroup_structure=True,
                   overdispersion_nu=0
                  ):
    """
    n: number of participants
    M_G: number of genetic features
    M_C: number of clinical features
    rank: number of subgroups
    seed: random seed
    gamma: ratio of marginal variance of covariate vs. subgroup component for G
    noise: weighting of random noise
    sparsity: controls sparsity of H_G and H_C
    rG: controls similarity between genetic subgroups
    rGC: controls similarity between corresponding clinical and genetic subgroups
    rZ: controls similarity between covariate and subgroup 1
    signed_cov_effects: no non-negative projection for U_G and U_C (bool)
    subgroup_structure: whether to include WHT in means of G and C (bool)
    overdispersion_nu: paramater to control overdispersion in negative binomial distribution for C, if !=0
    """

    rng = np.random.default_rng(seed)
    
    if subgroup_structure:
        # 1. Simulate W by random assignment to rank subgroups
        sizes = np.full(rank, n//rank) # evenly balance subgroups
        sizes[:n % rank] += 1
        labels = np.repeat(np.arange(rank), sizes)
        rng.shuffle(labels)
        W = np.zeros((n, rank), dtype=int)
        W[np.arange(n), labels] = 1
        
        # 2. Simulate H_G with genetic architecture similarity rG
        Sigma = (1-rG)*np.eye(rank,rank) + np.full((rank,rank),rG)
        H_G = proj_nonneg(rng.multivariate_normal(np.zeros((rank)), Sigma, size=M_G))
        
        # 3. Simulate H_C with similarity to genetic subgroups rGC
        H_C = proj_nonneg(make_correlated_matrices(rGC, H_G, seed=seed))
        
        # 4. impose sparsity (P(W_ij=0)=sparsity)
        mask = np.random.rand(*(M_C,rank)) > sparsity
        H_C = H_C * mask
        mask = np.random.rand(*(M_G,rank)) > sparsity
        H_G = H_G * mask
    
        # 5. Generate covariate with similarity to subgroup 1 structure
        z = make_correlated_matrices(rZ, W[:,1], y_nonneg=True, seed=seed)
    
    else: 
        assert rZ==0, "set rZ to 0 if no subgroup structure"
        z = proj_nonneg(rng.normal(size=n))
    
    # 5. Generate covariate matrix
    assert (z>=0).all()
    Z = np.column_stack([np.ones(len(z)), z])
    
    # 6. Simulate covariate effects
    U_C = rng.normal(size=(M_C, Z.shape[1]))
    U_G = rng.normal(size=(M_G, Z.shape[1]))
    if not signed_cov_effects:
        U_C = proj_nonneg(U_C)
        U_G = proj_nonneg(U_G)
    
    # 7. Generate matrix means
    if subgroup_structure:
        mu_C = W@H_C.T + Z@U_C.T + (noise)*proj_nonneg(rng.normal(size=(n, M_C)))
        # standardize A and B to have same marginal standard deviation
        A = W @ H_G.T
        B = Z @ U_G.T
        mu_G = A + gamma*(np.std(A) / np.std(B))*B + (noise)*proj_nonneg(rng.normal(size=(n, M_G)))
    else:
        mu_C = Z@U_C.T + (noise)*proj_nonneg(rng.normal(size=(n, M_C)))
        mu_G = Z@U_G.T + (noise)*proj_nonneg(rng.normal(size=(n, M_G)))
    
    if signed_cov_effects:
        mu_C = np.exp(mu_C)
        mu_G = np.exp(mu_G)
    
    # Generate the two observed matrices
    G = rng.poisson(mu_G).astype(float)  
    if overdispersion_nu==0:
        C = rng.poisson(mu_C).astype(float) 
    else:
        p_nb = mu_C / (mu_C + overdispersion_nu**2 * mu_C**2)
        n_nb = 1 / overdispersion_nu**2
        C = rng.poisson(n_nb, p_nb).astype(float) 
    if subgroup_structure:
        return_dict = {"G": G, "C": C, "Z":Z, "W_C": W, "H_G": H_G, "H_C": H_C, "U_G": U_G, "U_C": U_C}
    else:
        return_dict = {"G": G, "C": C, "Z":Z, "U_G": U_G, "U_C": U_C}
    return return_dict

In [20]:
simulate_views(n=600, M_G=20, M_C=20, 
                   rank=3, seed=0, gamma=1, noise=0,
                   sparsity=0, rG=0, rGC=0, rZ=0,
                   signed_cov_effects=True,subgroup_structure=False,
                   overdispersion_nu=0
                  )

{'G': array([[0., 1., 1., ..., 0., 1., 0.],
        [0., 0., 0., ..., 0., 5., 1.],
        [1., 0., 0., ..., 1., 1., 1.],
        ...,
        [1., 2., 0., ..., 1., 2., 0.],
        [1., 1., 1., ..., 0., 1., 3.],
        [1., 0., 2., ..., 1., 0., 0.]], shape=(600, 20)),
 'C': array([[0., 3., 0., ..., 1., 9., 4.],
        [1., 0., 2., ..., 2., 5., 3.],
        [0., 1., 4., ..., 3., 6., 1.],
        ...,
        [0., 0., 2., ..., 2., 7., 4.],
        [1., 3., 0., ..., 1., 3., 3.],
        [0., 0., 1., ..., 1., 7., 3.]], shape=(600, 20)),
 'Z': array([[1.        , 0.12573022],
        [1.        , 0.        ],
        [1.        , 0.64042265],
        ...,
        [1.        , 0.        ],
        [1.        , 0.        ],
        [1.        , 0.05103395]], shape=(600, 2)),
 'U_G': array([[-0.54616882,  0.31482097],
        [-0.60581285, -0.57312922],
        [-0.60774063, -2.2954242 ],
        [ 0.10549948, -1.26383536],
        [-0.10726912,  1.44996878],
        [-0.52100032, -0.542009